In [ ]:
import numpy as np
import faiss
import sqlite3
import pandas as pd

In [52]:
# Connexion à la base de données SQLite
conn = sqlite3.connect('../data/Y-love.db')  # Se connecter à la base de données
cur = conn.cursor()  # Créer un curseur

In [53]:
def moyenne_ponderee_embeddings(vecs, poids=None):
    if poids is None:
        return np.mean(vecs, axis=0)

    poids = np.array(poids).reshape(-1, 1)
    return np.sum(vecs * poids, axis=0) / np.sum(poids)

In [54]:
def get_vecteurs_Profil_User(user_id: str, conn=conn):
    df_vecteurs_profil_user = pd.read_sql_query("""
                SELECT E.categorie, E.text_initial, E.vecteur FROM Embeddings AS E 
                JOIN Profil_Embeddings AS PE ON E.embeddings_id = PE.embeddings_id
                JOIN Users AS U ON PE.user_id = U.user_id 
                WHERE U.user_id = '""" + user_id + "';"
                , conn)
    # Convertir les vecteurs stockés en BLOB en tableaux numpy
    df_vecteurs_profil_user['vecteur'] = df_vecteurs_profil_user['vecteur'].apply(lambda x: np.frombuffer(x, dtype=np.float32))
    
    # Tableaux avec tous les textes
    liste_hobby = df_vecteurs_profil_user[df_vecteurs_profil_user['categorie'] == 'hobby']['text_initial'].to_list()
    liste_trait = df_vecteurs_profil_user[df_vecteurs_profil_user['categorie'] == 'trait']['text_initial'].to_list()
    liste_job = df_vecteurs_profil_user[df_vecteurs_profil_user['categorie'] == 'metier']['text_initial'].to_list()
    
    # Calculer les vecteurs moyens pondérés pour chaque catégorie
    moy_vect_hobby = moyenne_ponderee_embeddings(df_vecteurs_profil_user[df_vecteurs_profil_user['categorie'] == 'hobby']['vecteur'].to_list())
    moy_vect_trait = moyenne_ponderee_embeddings(df_vecteurs_profil_user[df_vecteurs_profil_user['categorie'] == 'trait']['vecteur'].to_list())
    moy_vect_job = moyenne_ponderee_embeddings(df_vecteurs_profil_user[df_vecteurs_profil_user['categorie'] == 'metier']['vecteur'].to_list()) # Je le fait pour le type de sortie str -> np.array
    
    # Retourner les vecteurs moyens et le vecteur métier
    return liste_hobby, moy_vect_hobby, liste_trait, moy_vect_trait, liste_job, moy_vect_job

In [55]:
def normalisation_des_vecteurs(vecteurs: pd.Series):
    X = np.vstack(vecteurs).astype(np.float32)
    faiss.normalize_L2(X)
    return list(X)

In [56]:
# Récupération des données des utilisateurs
df_Users = pd.read_sql_query("SELECT * FROM Users", conn)
print(df_Users.shape)
df_Users.head()

(10002, 7)


,user_id,email,password_hash,nom,prenom,age,genre
0,0dbc820e-f937-432f-8895-2eec661d146a,maggie.riou@mail.com,6c52b7d85ee1d2551aaed4b5a962eb9298a0e67daeace3...,Riou,Maggie,73,M
1,cc044731-5144-4ad0-9bb8-84365245a684,céline.letellier@mail.com,e7d444f12372dc74bbe55d1aea4cfa9df9f232a85e2563...,Letellier,Céline,54,M
2,18e6c253-149b-453d-88b8-9932c69cd3c7,stéphane.mary@mail.com,aebac53c46bbeff10fdd26ca0e2196a9bfc1d19bf88eb1...,Mary,Stéphane,74,F
3,64788fa5-dea0-4e3e-b350-c50484d3143c,margaud.renaud@mail.com,8523e8c62e115d851a68beecf6c0b8386a1867afd1032b...,Renaud,Margaud,65,M
4,8f57e118-2708-4482-9317-da5131633ee0,bertrand.techer@mail.com,894e19ee48abc93a803b06b2a9ab7bd867480e8e4a8172...,Techer,Bertrand,33,M


## Mise en forme des données
---

In [57]:
# On garde uniquement les 10 000 premiers utilisateurs
df_Users = df_Users[:10_000] 

In [58]:
# Calculer et ajouter les vecteurs de profil pour chaque utilisateur
df_Users["liste_hobby"], df_Users["vecteur_hobby"], df_Users["liste_trait"], df_Users["vecteur_trait"], df_Users["liste_metier"], df_Users["vecteur_metier"] = zip(*df_Users["user_id"].apply(get_vecteurs_Profil_User))
# Temps (10_000) : 3m

In [59]:
# Normalisation des vecteurs
df_Users["vecteur_hobby"] = normalisation_des_vecteurs(df_Users["vecteur_hobby"].values)
df_Users["vecteur_trait"] = normalisation_des_vecteurs(df_Users["vecteur_trait"].values)
df_Users["vecteur_metier"] = normalisation_des_vecteurs(df_Users["vecteur_metier"].values)
# Temps (10_000) : 1s

In [ ]:
# Pas forcement utile car on filtre avant le matching
# Tokenisation de la colonne genre
# df_Users['genre_token'] = df_Users['genre'].apply(lambda x: 1 if x == 'F' else 0)
# Temps (10_000) : 0.5s

In [61]:
# Ajout d'un identifiant FAISS pour chaque utilisateur car FAISS nécessite des identifiants entiers
df_Users["faiss_id"] = df_Users.index.astype(np.int64)

In [62]:
print(df_Users.dtypes)
print()
print(df_Users.shape)
df_Users.head()

user_id           object
email             object
password_hash     object
nom               object
prenom            object
age                int64
genre             object
liste_hobby       object
vecteur_hobby     object
liste_trait       object
vecteur_trait     object
liste_metier      object
vecteur_metier    object
genre_token        int64
faiss_id           int64
dtype: object

(10000, 15)


,user_id,email,password_hash,nom,prenom,age,genre,liste_hobby,vecteur_hobby,liste_trait,vecteur_trait,liste_metier,vecteur_metier,genre_token,faiss_id
0,0dbc820e-f937-432f-8895-2eec661d146a,maggie.riou@mail.com,6c52b7d85ee1d2551aaed4b5a962eb9298a0e67daeace3...,Riou,Maggie,73,M,"[BD, Bricolage, Bachata, Sneakers, Spéléologie]","[-0.03862239, 0.061011598, -0.0016667395, 0.00...",[Généreux],"[0.0048402036, 0.05863952, -0.048253857, -0.00...",[Aide-soignant],"[-0.0075356853, -0.06604484, 0.028867722, 0.06...",0,0
1,cc044731-5144-4ad0-9bb8-84365245a684,céline.letellier@mail.com,e7d444f12372dc74bbe55d1aea4cfa9df9f232a85e2563...,Letellier,Céline,54,M,"[Danse classique, Contrebasse, Création de jeux]","[0.04034124, -0.024886008, -0.0046114777, 0.00...","[Manipulateur, Travailleur, Réfléchi]","[-0.03645362, 0.028777169, -0.03302835, 0.0377...",[Chef d'orchestre],"[0.045273755, 0.027367871, 0.049840022, -0.016...",0,1
2,18e6c253-149b-453d-88b8-9932c69cd3c7,stéphane.mary@mail.com,aebac53c46bbeff10fdd26ca0e2196a9bfc1d19bf88eb1...,Mary,Stéphane,74,F,"[Aquariophilie, Natation, Sciences, Cocktails,...","[0.030032584, 2.5676325e-06, 0.029630832, 0.04...","[Introverti, Déterminé, Responsable, Narcissique]","[0.03367659, 0.033011634, -0.046546932, -0.001...",[Formateur],"[-0.0045083635, 0.017517986, -0.04249893, -0.0...",1,2
3,64788fa5-dea0-4e3e-b350-c50484d3143c,margaud.renaud@mail.com,8523e8c62e115d851a68beecf6c0b8386a1867afd1032b...,Renaud,Margaud,65,M,"[Mycologie, Vegan]","[0.011233915, 0.005014991, -0.006683798, 0.015...",[Innovant],"[-0.015062465, 0.01620782, -0.07769034, -0.042...",[UX designer],"[-0.02284487, 0.02877729, 0.026684808, 0.00260...",0,3
4,8f57e118-2708-4482-9317-da5131633ee0,bertrand.techer@mail.com,894e19ee48abc93a803b06b2a9ab7bd867480e8e4a8172...,Techer,Bertrand,33,M,"[Puzzles, Animation]","[-0.0490692, -0.062395357, 0.012919333, 0.0300...","[Manipulateur, Innovant, Conservateur, Intolér...","[-0.019448599, 0.036043942, -0.0719021, -0.022...",[Directeur artistique],"[0.035995476, 0.030553317, 0.023541685, 0.0790...",0,4


## Création du modèle
---

In [63]:
user_id = "a52f0e58-a88c-4098-846b-5af80398ac87" # Yann
# user_id = "de7e320d-00f3-4161-a7dc-eb78ea6e92a9" # Elodie

In [64]:
# Récupération des données des utilisateurs
df_Envie_Users = pd.read_sql_query("SELECT E.user_id, E.envies_id, E.Age_min, E.Age_max, E.genre FROM Users AS U JOIN Envies AS E ON U.user_id = E.user_id WHERE U.user_id = '" + user_id + "'", conn)
print(df_Envie_Users.shape)
df_Envie_Users.head()

(1, 5)


,user_id,envies_id,Age_min,Age_max,genre
0,a52f0e58-a88c-4098-846b-5af80398ac87,1,20,25,F


In [65]:
def get_vecteurs_Envie_User(user_id: str, conn=conn):
    df_vecteurs_envie_user = pd.read_sql_query("""
                SELECT E.categorie, E.text_initial, E.vecteur FROM Embeddings AS E 
                JOIN Envies_Embeddings AS EE ON E.embeddings_id = EE.embeddings_id
                JOIN Envies AS U ON EE.envies_id = U.envies_id 
                WHERE U.user_id = '""" + user_id + "';"
                , conn)
    # Convertir les vecteurs stockés en BLOB en tableaux numpy
    df_vecteurs_envie_user['vecteur'] = df_vecteurs_envie_user['vecteur'].apply(lambda x: np.frombuffer(x, dtype=np.float32))
    
    # Tableaux avec tous les textes
    liste_hobby = df_vecteurs_envie_user[df_vecteurs_envie_user['categorie'] == 'hobby']['text_initial'].to_list()
    liste_trait = df_vecteurs_envie_user[df_vecteurs_envie_user['categorie'] == 'trait']['text_initial'].to_list()
    liste_job = df_vecteurs_envie_user[df_vecteurs_envie_user['categorie'] == 'metier']['text_initial'].to_list()
    
    # Calculer les vecteurs moyens pondérés pour chaque catégorie
    moy_vect_hobby = moyenne_ponderee_embeddings(df_vecteurs_envie_user[df_vecteurs_envie_user['categorie'] == 'hobby']['vecteur'].to_list())
    moy_vect_trait = moyenne_ponderee_embeddings(df_vecteurs_envie_user[df_vecteurs_envie_user['categorie'] == 'trait']['vecteur'].to_list())
    moy_vect_job = moyenne_ponderee_embeddings(df_vecteurs_envie_user[df_vecteurs_envie_user['categorie'] == 'metier']['vecteur'].to_list()) # Je le fait pour le type de sortie str -> np.array
    
    # Retourner les vecteurs moyens et le vecteur métier
    return liste_hobby, moy_vect_hobby, liste_trait, moy_vect_trait, liste_job, moy_vect_job

#### Nettoyage du vecteur d'envie de l'utilisateur
---

Pour qu'il correspond aux vecteurs utilisateur

In [66]:
# Calculer et ajouter les vecteurs d'envie de l'utilisateur
df_Envie_Users["liste_hobby"], df_Envie_Users["vecteur_hobby"], df_Envie_Users["liste_trait"], df_Envie_Users["vecteur_trait"], df_Envie_Users["liste_metier"], df_Envie_Users["vecteur_metier"] = zip(*df_Envie_Users["user_id"].apply(get_vecteurs_Envie_User))

# Normalisation des vecteurs
df_Envie_Users["vecteur_hobby"] = normalisation_des_vecteurs(df_Envie_Users["vecteur_hobby"].values)
df_Envie_Users["vecteur_trait"] = normalisation_des_vecteurs(df_Envie_Users["vecteur_trait"].values)
df_Envie_Users["vecteur_metier"] = normalisation_des_vecteurs(df_Envie_Users["vecteur_metier"].values)

# Pas forcement utile car on filtre avant le matching
# Tokenisation de la colonne genre
# df_Envie_Users['genre_token'] = df_Envie_Users['genre'].apply(lambda x: 1 if x == 'F' else 0)

In [67]:
print(df_Envie_Users.dtypes)
print()
print(df_Envie_Users.shape)
df_Envie_Users.head()

user_id           object
envies_id          int64
Age_min            int64
Age_max            int64
genre             object
liste_hobby       object
vecteur_hobby     object
liste_trait       object
vecteur_trait     object
liste_metier      object
vecteur_metier    object
dtype: object

(1, 11)


,user_id,envies_id,Age_min,Age_max,genre,liste_hobby,vecteur_hobby,liste_trait,vecteur_trait,liste_metier,vecteur_metier
0,a52f0e58-a88c-4098-846b-5af80398ac87,1,20,25,F,"[Intelligence artificielle, Sculpture, Golf, C...","[0.031128218, 0.043568797, -0.034548193, -0.02...","[Impulsif, Honnête, Enthousiaste, Motivé, Into...","[0.0492927, 0.033372696, -0.05250917, 0.006664...","[Livreur, Magasinier]","[-0.021540193, 0.040092025, -0.054153927, 0.07..."


## Fonction du modèle
---

On fait le filtre sur le genre et sur la tranche d'âge rechercher avant de faire le modèle

In [89]:
def search(index: faiss.IndexIDMap, envie_vec, k: int = 5) -> pd.DataFrame:
    q = envie_vec.astype(np.float32).reshape(1, -1)

    # k nearest neighbors (run du modèle Faiss)
    D, I = index.search(q, k)

    # Récupérer les informations des utilisateurs correspondants
    results = df_Users.iloc[I[0]][["faiss_id", "nom", "prenom", "age", "genre", "liste_hobby", "liste_trait", "liste_metier"]].copy()
    results["score"] = D[0] # Ajouter les scores de similarité

    return results

In [90]:
# Filtrage des utilisateurs selon les envies de l'utilisateur
df_filtrer = df_Users[df_Users["genre"] == df_Envie_Users["genre"].values[0]]
df_filtrer = df_filtrer[df_filtrer["age"] <= df_Envie_Users["Age_max"].values[0]]
df_filtrer = df_filtrer[df_filtrer["age"] >= df_Envie_Users["Age_min"].values[0]]
print(df_filtrer.shape)
df_filtrer.head()

(481, 15)


,user_id,email,password_hash,nom,prenom,age,genre,liste_hobby,vecteur_hobby,liste_trait,vecteur_trait,liste_metier,vecteur_metier,genre_token,faiss_id
5,237e1d12-d072-4cf7-bd4a-3d55f051d180,philippine.neveu@mail.com,6e9c2d0ffa81c4fe9fd9b76a66a9193138ed756b78580a...,Neveu,Philippine,21,F,[Aquarelle],"[0.06516809, -0.01756409, 0.037992906, -0.0176...","[Intolérant, Perfectionniste, Prudent, Prudent...","[0.01941232, 0.03878159, -0.04482321, 0.005047...",[Garde d'enfants],"[0.025441665, 0.06148834, -0.01736276, 0.02416...",1,5
16,ca11d347-65a9-4d73-a76a-8c6a7985230e,emmanuelle.goncalves@mail.com,5d2f2b41dcba04e00196b4d82fc3cf3a85be28414da8b4...,Goncalves,Emmanuelle,24,F,"[Planche à voile, Taekwondo, Clarinette, Théât...","[0.013220993, 0.014447479, -0.04153922, -0.001...","[Égocentrique, Introverti, Autonome, Rigoureux...","[0.017890478, 0.00810705, -0.007077124, -0.010...",[Conducteur de train],"[0.010747934, -0.04006036, -0.018779036, 0.028...",1,16
17,943ab061-28ec-4348-81f8-a70b9a01fa53,émile.dupuy@mail.com,5a60cec87c4e703602d0ee128bc015116bdcbf2c09fefb...,Dupuy,Émile,21,F,"[Mangas, Motocross, Motocross]","[-0.031968664, 0.08329086, -0.06342403, 0.0290...","[Courageux, Innovant]","[0.01386365, 0.07428072, -0.10257193, -0.01698...",[Chauffagiste],"[3.864274e-05, 0.016669912, -0.026264591, 0.05...",1,17
26,7dc84720-47a9-440f-9575-fa27d57053c4,alice.munoz@mail.com,62f32e9a96dac6ba51de458a6d62c38cb3fb67ad2afa2c...,Munoz,Alice,20,F,"[Astrophotographie, Électronique, Survie]","[0.01533256, 0.06672423, -0.0038313882, 0.0267...",[Endurant],"[0.069972984, 0.06730721, -0.046610687, 0.0545...",[Chauffeur routier],"[-0.0056866254, 0.05025225, -0.04505707, 0.051...",1,26
64,e6c026e9-d378-4133-9337-dc1773d3f187,michelle.lecomte@mail.com,2ac9a470216c0f170b55c47d0ee9c2b908e3cb4ebe884b...,Lecomte,Michelle,24,F,"[Sound design, Sculpture, Course à pied, Danse...","[0.053198755, 0.020489715, 0.022166202, 0.0125...","[Perfectionniste, Juste]","[0.029011304, 0.009723752, -0.009275548, -0.02...",[Développeur web],"[-0.040696584, -0.046534024, -0.0016830092, -0...",1,64


#### Modèle des hobbies
---

In [91]:
# Creation de l'index Faiss avec les vecteurs filtrés
index_hobby = faiss.IndexIDMap(faiss.IndexFlatIP(384))

X_hobby = np.vstack(df_filtrer["vecteur_hobby"].values).astype(np.float32)
index_hobby.add_with_ids(X_hobby, df_filtrer["faiss_id"].values)

In [92]:
res_hobby = search(index_hobby, df_Envie_Users["vecteur_hobby"].values[0], k=10)
print(res_hobby.shape)
res_hobby.head()

(10, 9)


,faiss_id,nom,prenom,age,genre,liste_hobby,liste_trait,liste_metier,score
6321,6321,Pages,Christophe,21,F,"[Électronique, Clarinette, Tissage, Course à p...",[Altruiste],[Électricien],0.846127
9948,9948,Guillou,Éric,25,F,"[Broderie, Tissage]","[Pragmatique, Enthousiaste, Pessimiste, Travai...",[Pentester],0.843040
9254,9254,Pottier,Manon,25,F,"[DIY, Voile, Saxophone, Psychologie, Voyage]","[Ambitieux, Humble]",[Vidéaste],0.833873
7803,7803,Leclerc,Sylvie,21,F,"[Voyage, Boulangerie, Nouvelles, Bodyboard, Pa...","[Vaniteux, Bienveillant, Motivé, Prudent]",[Analyste financier],0.827507
8876,8876,Fernandes,Nathalie,23,F,[Broderie],"[Introverti, Créatif, Impulsif, Honnête]",[Assistant de gestion],0.826718


#### Modèle des traits de caractères
---

In [93]:
# Creation de l'index Faiss avec les vecteurs filtrés
index_traits = faiss.IndexIDMap(faiss.IndexFlatIP(384))

X_traits = np.vstack(df_filtrer["vecteur_trait"].values).astype(np.float32)
index_traits.add_with_ids(X_traits, df_filtrer["faiss_id"].values)

In [94]:
res_traits = search(index_traits, df_Envie_Users["vecteur_trait"].values[0], k=10)
print(res_traits.shape)
res_traits.head()

(10, 9)


,faiss_id,nom,prenom,age,genre,liste_hobby,liste_trait,liste_metier,score
3865,3865,Maréchal,Jules,22,F,"[Mots croisés, Marche nordique]","[Optimiste, Négatif, Impulsif, Juste]",[Ingénieur IA],0.953604
4689,4689,Lombard,Michel,25,F,"[Bricolage, Calligraphie, Menuiserie, Graffiti]","[Négatif, Idéaliste, Enthousiaste, Contemplati...",[Architecte logiciel],0.949496
9668,9668,Marchand,Alexandrie,25,F,[Rugby],"[Fainéant, Enthousiaste]",[Auditeur],0.941947
8876,8876,Fernandes,Nathalie,23,F,[Broderie],"[Introverti, Créatif, Impulsif, Honnête]",[Assistant de gestion],0.939683
2771,2771,Payet,Véronique,24,F,[Football],"[Impulsif, Pessimiste, Discret, Généreux, Égoc...",[Administrateur réseaux],0.933250


#### Modèle des métiers
---

In [95]:
# Creation de l'index Faiss avec les vecteurs filtrés
index_metier = faiss.IndexIDMap(faiss.IndexFlatIP(384))

X_metier = np.vstack(df_filtrer["vecteur_metier"].values).astype(np.float32)
index_metier.add_with_ids(X_metier, df_filtrer["faiss_id"].values)

In [96]:
res_metier = search(index_metier, df_Envie_Users["vecteur_metier"].values[0], k=10)
print(res_metier.shape)
res_metier.head()

(10, 9)


,faiss_id,nom,prenom,age,genre,liste_hobby,liste_trait,liste_metier,score
3153,3153,Martin,Christiane,24,F,"[Ornithologie, Kitesurf]","[Enthousiaste, Agressif, Agressif, Altruiste]",[Magasinier],0.914897
2231,2231,Techer,Patricia,20,F,"[Guitare, Ski alpin, Alto, Programmation]","[Courageux, Agressif, Envieux, Courageux, Cont...",[Magasinier],0.914897
606,606,Bigot,Suzanne,20,F,"[Fitness, Jeux rétro, Généalogie, Impression 3D]","[Égocentrique, Instable, Motivé]",[Magasinier],0.914897
9210,9210,Descamps,Simone,24,F,"[Vegan, Danse, Origami, Fermentation, Marche n...","[Endurant, Manipulateur, Intolérant]",[Livreur],0.871867
8904,8904,Seguin,Thibaut,21,F,"[Sneakers, Casse-têtes]",[Juste],[Livreur],0.871867


#### Concaténation des resultats
---

In [97]:
df_resultats_final = pd.concat([res_hobby, res_traits, res_metier], axis=0, ignore_index=True)
df_resultats_final.drop_duplicates(subset=["faiss_id"], keep="first", inplace=True)
df_resultats_final = df_resultats_final.sort_values(by="score", ascending=False).reset_index(drop=True)
print(df_resultats_final.shape)
df_resultats_final.head()

(29, 9)


,faiss_id,nom,prenom,age,genre,liste_hobby,liste_trait,liste_metier,score
0,3865,Maréchal,Jules,22,F,"[Mots croisés, Marche nordique]","[Optimiste, Négatif, Impulsif, Juste]",[Ingénieur IA],0.953604
1,4689,Lombard,Michel,25,F,"[Bricolage, Calligraphie, Menuiserie, Graffiti]","[Négatif, Idéaliste, Enthousiaste, Contemplati...",[Architecte logiciel],0.949496
2,9668,Marchand,Alexandrie,25,F,[Rugby],"[Fainéant, Enthousiaste]",[Auditeur],0.941947
3,2771,Payet,Véronique,24,F,[Football],"[Impulsif, Pessimiste, Discret, Généreux, Égoc...",[Administrateur réseaux],0.933250
4,9277,Fleury,Gabriel,21,F,"[Céramique, Théâtre, Brassage de bière, Synthé...","[Discret, Indépendant, Anxieux, Indécis, Sincère]",[Huissier de justice],0.930087
